###  ModelCallLimitMiddleware中间件
限制模型调用次数，避免无限循环，控制调用成本。


In [1]:
# 1、模型的初始化
import os
from dotenv import load_dotenv
from langchain_qwq import ChatQwen

# 从.env文件中加载环境变量
load_dotenv(override=True)
# 模型的初始化
model = ChatQwen(
    model="qwen3.6-27b",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
)

In [ ]:
# 优雅推出
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from typing import List

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2,  # 每个线程最多2次模型调用
            # run_limit=5,  # 每次运行最多5次
            exit_behavior="end",  # 达到限制后退出
        ),
    ],
)


def pretty_iterate_msg(
    messages: List[SystemMessage | HumanMessage | AIMessage | ToolMessage],
):
    for msg in messages:
        msg.pretty_print()


config = {"configurable": {"thread_id": "1"}}

response_first = agent.invoke(
    {"messages": [HumanMessage("你好")]},
    config=config,
)
print("=" * 30, "> first <", "=" * 30)
pretty_iterate_msg(response_first["messages"])

response_second = agent.invoke(
    {"messages": [HumanMessage("你是谁？")]},
    config=config,
)
print("=" * 30, "> second <", "=" * 30)
pretty_iterate_msg(response_second["messages"])

response_third = agent.invoke(
    {"messages": [HumanMessage("你能帮我做什么？")]},
    config=config,
)
print("=" * 30, "> third <", "=" * 30)
pretty_iterate_msg(response_third["messages"])

============================== > first < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！有什么我可以帮你的吗？
============================== > second < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！有什么我可以帮你的吗？
================================ Human Message =================================

你是谁？
================================== Ai Message ==================================

我是 Qwen（通义千问），由阿里巴巴集团旗下通义实验室自主研发的大语言模型。有什么我可以帮你的吗？
============================== > third < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！有什么我可以帮你的吗？
=================

In [3]:
# 整个会话限制-抛异常
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from typing import List

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2,
            # run_limit=5,
            exit_behavior="error",
        ),
    ],
)


def pretty_iterate_msg(
    messages: List[SystemMessage | HumanMessage | AIMessage | ToolMessage],
):
    for msg in messages:
        msg.pretty_print()


config = {"configurable": {"thread_id": "1"}}

response_first = agent.invoke(
    {"messages": [HumanMessage("你好")]},
    config=config,
)
print("=" * 30, "> first <", "=" * 30)
pretty_iterate_msg(response_first["messages"])

response_second = agent.invoke(
    {"messages": [HumanMessage("你是谁？")]},
    config=config,
)
print("=" * 30, "> second <", "=" * 30)
pretty_iterate_msg(response_second["messages"])

response_third = agent.invoke(
    {"messages": [HumanMessage("你能帮我做什么？")]},
    config=config,
)
print("=" * 30, "> third <", "=" * 30)
pretty_iterate_msg(response_third["messages"])

============================== > first < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！有什么我可以帮你的吗？
============================== > second < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！有什么我可以帮你的吗？
================================ Human Message =================================

你是谁？
================================== Ai Message ==================================

我是 Qwen（通义千问），由阿里巴巴集团旗下通义实验室自主研发的大语言模型。很高兴为你服务！有什么我可以帮你的吗？


ModelCallLimitExceededError: Model call limits exceeded: thread limit (2/2)